In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from bhtrace.geometry import Photon, Observer
import bhtrace.geometry.spacetime as st
from bhtrace.tracing import PTracer

import bhtrace.medium as bhM
import bhtrace.graphics as bhg
import bhtrace.utils.units as bhU

from bhtrace.grrt.runner import GRRT
from bhtrace.grrt.radiation import IntegralFlux

In [ ]:
N = 256  # Image resolution
D = 32
KERR_A = 0.90
OBS_R = 20.0
OBS_INCLINATION = torch.pi / 180 * 84
TRACE_T = 36
TRACE_NSTEPS = 256
TRACE_R_MAX = 25
TRACE_EPS = 1e-5

BASE = st.KerrBL(a=KERR_A)

photon = Photon(spacetime=BASE)
obs = Observer(
    spacetime=BASE,
    r=OBS_R,
    inclination=OBS_INCLINATION,
    u=torch.Tensor([1, 0, 0, 0]),
)
obs.generate_net(net_shape="square", net_rng=(N, N), net_size=(D, D))
X0, P0 = obs.setup_ic(photon)

tracer = PTracer(ode_method="VCABM4")

dev = "cuda" if torch.cuda.is_available() else 'cpu'
print(f"Using device: {dev}")

# --- Running --- 
tracer.__const_dx__ = True
traj = tracer.forward(
    photon, X0, P0, T=TRACE_T, nsteps=TRACE_NSTEPS, r_max=TRACE_R_MAX, eps=TRACE_EPS, device=dev, dtype=torch.float64
)

In [ ]:
medium = bhM.KeplerianDisk(BASE, bhU.m_sgrA , 5e-5, clockwise=True)
model = IntegralFlux()

grrt = GRRT(medium=medium)
grrt.set_models(model)

shoot = grrt.compute(traj, dtype=torch.float32, device=dev)

In [ ]:
# === Plot image ===

image = model.intensities.x.cpu()
image = image.clip(max=image.max()*0.9)

vmin = image.min()
vmax = image.max()

fig, ax = plt.subplots(1, 1, figsize=(10, 10))

image_data = image.view(N, N).cpu().transpose(-1, -2)
im = ax.imshow(image_data, cmap="afmhot", vmin=vmin, vmax=vmax)
fig.colorbar(im, ax=ax)
plt.show()

In [ ]:
# Trivial example for accounting finite VLBI resolution

import torchvision.transforms.functional as TF

image_width = bhU.m_sgrA * bhU.G / bhU.c.pow(2) * D / bhU.D_sgrA
print(image_width / bhU.muarcsec)
pixel_size = image_width.f / N
angular_FWHM = 25 * bhU.muarcsec / 4
sigma_pix = (angular_FWHM / pixel_size)   # example value in pixels
print(sigma_pix)
sigma = sigma_pix.f

kernel_size = int(2 * round(3 * sigma) + 1)  # odd size, ~3 sigma radius

blurred = TF.gaussian_blur(image_data.unsqueeze(0).cuda(), kernel_size=[kernel_size, kernel_size], sigma=[sigma, sigma]).cpu()

vmin = blurred.min()
vmax = blurred.max()

fig, ax = plt.subplots(1, 1, figsize=(10, 10))

im = ax.imshow(blurred.squeeze().cpu().numpy(), cmap="afmhot", vmin=vmin, vmax=vmax)
fig.colorbar(im, ax=ax)
plt.show()